# **Ejercicio 2: Entregable Final (videogames_reviews.csv)**

Vamos a trabajar con un dataframe que contiene información sobre reseñas de videojuegos. Cada fila del dataframe contiene la información de una reseña, y las columnas son las siguientes:
- Contenido: texto de la reseña
- Valoración: Recomendado o No recomendado
- Recomendado binario: 1 si la valoración es Recomendado, 0 si la valoración es No recomendado
  
El objetivo de este ejercicio es conseguir extraer información valiosa de las reseñas de un dataset más 'crudo' o que necesita un preprocesado mayor que el del ejercicio 1.

1. **Filtramos por las 100 reseñas con el contenido más largo**. Esto ya os lo doy hecho más abajo.
2. En el DataFrame resultante, habremos obtenido muchas filas en las que realmente el contenido de la reseña no es nada relevante. **Instruir vía Prompting a un LLM para que filtre las filas del DataFrame que de verdad aportan información de la que se pueden extraer insights relevantes**
3. **Paso final:** extraer información relevante de este último dataframe doblemente filtrado usando un segundo LLM. Por ejemplo, un json final como este:

    ```json
    {
    "Id": "{{id}}",
    "SentimientoGeneral": "Positivo|Negativo|Neutral",
    "AspectosPositivos": "[lista de aspectos positivos mencionados]",
    "AspectosNegativos": "[lista de aspectos negativos mencionados]",
    "Dificultad": "Demasiado Fácil|Fácil|Equilibrado|Difícil|Demasiado Difícil|No Mencionado",
    "Recomendado": "{{recomendado}}"
    }
    ```

**Consideraciones adicionales:**

- También tienes un csv de ejemplo de resultado final (analisis_videojuegos_resultados.csv) donde se han extraído otras entidades y se ha entregado otro formato. **Esto es a libre elección, pero un mínimo de 3 entidades extraídas** por el segundo LLM.
- Los modelos suelen ser buenos realizando tareas de una en una. Por eso separamos en dos pasos el filtrado de contenido relevante y el de extracción de ese contenido.
- Puedes usar cualquier proveedor y canal: vía API (como Groq/Gemini) o en local vía HuggingFce teniendo en cuenta las limitaciones y ventajas de cada uno, rate limits del modelo elegido, tamaño... 
- Tener en cuenta qué cantidad de tokens/información/tareas le estamos pasando al LLM en cada llamada comparándolo con los límites que tenemos para el LLM elegido
- Se valorará **justificar cada elección** como: cada time.stop() añadido, cada lote de filas de DataFrame que pasemos en cada llamada... etc ajustándonos como digo a esos límites que tenemos.

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('videogames_reviews.csv')
data

,Unnamed: 0,Contenido,Valoración,Recomendado_binario
0,0,2 marzo so bad,No recomendado,0
1,1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0
3,3,the world in this game is extremely static the...,No recomendado,0
4,4,zero replayability i finished this game in abo...,No recomendado,0
...,...,...,...,...
19995,19995,si,Recomendado,1
19996,19996,cojonudo,Recomendado,1
19997,19997,reostia historia guapisima graficos impresiona...,Recomendado,1
19998,19998,basicamente sublime obra maestra,Recomendado,1


In [3]:
# Análisis exploratorio básico
print("Información básica del dataframe:")
print(f"Forma del dataframe: {data.shape}")
print(f"Columnas: {list(data.columns)}")
print("\nTipos de datos:")
print(data.dtypes)
print("\nPrimeras filas:")
data.head()

Información básica del dataframe:
Forma del dataframe: (20000, 4)
Columnas: ['Unnamed: 0', 'Contenido', 'Valoración', 'Recomendado_binario']

Tipos de datos:
Unnamed: 0             int64
Contenido                str
Valoración               str
Recomendado_binario    int64
dtype: object

Primeras filas:


,Unnamed: 0,Contenido,Valoración,Recomendado_binario
0,0,2 marzo so bad,No recomendado,0
1,1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0
3,3,the world in this game is extremely static the...,No recomendado,0
4,4,zero replayability i finished this game in abo...,No recomendado,0


In [4]:
# Análisis de valores nulos y únicos
print("Valores nulos por columna:")
print(data.isnull().sum())
print("\nValores únicos en 'Valoración':")
print(data['Valoración'].value_counts())
print("\nEstadísticas de la columna 'Recomendado_binario':")
print(data['Recomendado_binario'].value_counts())

Valores nulos por columna:
Unnamed: 0               0
Contenido              288
Valoración               0
Recomendado_binario      0
dtype: int64

Valores únicos en 'Valoración':
Valoración
No recomendado    10000
Recomendado       10000
Name: count, dtype: int64

Estadísticas de la columna 'Recomendado_binario':
Recomendado_binario
0    10000
1    10000
Name: count, dtype: int64


## **PASO 1**
Obtener los 100 comentarios con más texto para análisis posterior.

In [5]:
# Preprocesamiento simplificado: Top 100 comentarios con más texto
print("Preprocesando datos para obtener los 100 comentarios más largos...")

# Eliminar filas con contenido nulo
df_limpio = data.dropna(subset=['Contenido']).copy()
print(f"Comentarios válidos (sin nulos): {len(df_limpio)}")

# Calcular longitud de caracteres
df_limpio['longitud_caracteres'] = df_limpio['Contenido'].str.len()

# Seleccionar los 100 comentarios más largos
df_top_100 = df_limpio.nlargest(100, 'longitud_caracteres').reset_index(drop=True)

# Eliminar columnas innecesarias
if 'Unnamed: 0' in df_top_100.columns:
    df_top_100 = df_top_100.drop('Unnamed: 0', axis=1)

print(f"\nDataset final: {len(df_top_100)} comentarios")
print(f"Longitud promedio: {df_top_100['longitud_caracteres'].mean():.1f} caracteres")
print(f"Longitud mínima: {df_top_100['longitud_caracteres'].min()} caracteres")
print(f"Longitud máxima: {df_top_100['longitud_caracteres'].max()} caracteres")

print("\nDistribución de valoraciones en el top 100:")
print(df_top_100['Valoración'].value_counts())

# Mostrar el dataframe resultante
df_top_100

Preprocesando datos para obtener los 100 comentarios más largos...
Comentarios válidos (sin nulos): 19712

Dataset final: 100 comentarios
Longitud promedio: 6772.1 caracteres
Longitud mínima: 5421 caracteres
Longitud máxima: 7972 caracteres

Distribución de valoraciones en el top 100:
Valoración
No recomendado    65
Recomendado       35
Name: count, dtype: int64


,Contenido,Valoración,Recomendado_binario,longitud_caracteres
0,suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii...,Recomendado,1,7972
1,this was probably my first preorder i felt tha...,Recomendado,1,7662
2,oh well admittedly its difficult for to write ...,No recomendado,0,7638
3,oh well admittedly its difficult for to write ...,No recomendado,0,7638
4,i know many will handwave away any criticisms ...,No recomendado,0,7609
...,...,...,...,...
95,4 febrero i was not seated in the glorious hyp...,Recomendado,1,5607
96,cyberpunk 2077cybermeme 2077what can there be ...,Recomendado,1,5573
97,brujo geralt riviatras recuperar memoria pasad...,Recomendado,1,5516
98,after hearing all the fuss about skyrim for ma...,No recomendado,0,5458


## Paso 2: filtrado de reseñas con un primer LLM

Uso Groq porque ya lo hemos visto en clase y es rápido. Descarté los modelos locales de Hugging Face porque en CPU serían muy lentos con textos de 7.000 caracteres.

Para el filtro uso gpt-oss-20b, que es pequeño y suficiente para decidir si una reseña sirve o no. Para la extracción del paso 3 uso gpt-oss-120b, que es más grande y se le da mejor una tarea con más campos.

La cuenta tiene un límite de 8.000 tokens por minuto (lo vi en las cabeceras de la respuesta de la API) y cada reseña ronda los 1.700 tokens. Por eso mando lotes de 2 reseñas, y después de cada llamada hago un time.sleep según los tokens gastados para no pasarme del límite. Uso temperature 0 para que el resultado sea siempre el mismo.

In [6]:
import os
import json
import time
from dotenv import load_dotenv
from groq import Groq

load_dotenv(".env")
client = Groq(api_key=os.getenv("GROQ_API"))

modelo_filtro = "openai/gpt-oss-20b"
modelo_extraccion = "openai/gpt-oss-120b"

batch_size = 2  # reseñas por llamada

In [7]:
# Función para llamar a Groq pidiendo la respuesta en JSON
def preguntar_groq(modelo, sys_prompt, prompt, max_tokens):
    for intento in range(3):
        try:
            response = client.chat.completions.create(
                model=modelo,
                messages=[
                    {"role": "system", "content": sys_prompt},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
                max_completion_tokens=max_tokens,
                reasoning_effort="low",  # los tokens de razonamiento también cuentan para el límite
                response_format={"type": "json_object"}
            )
            # Pausa proporcional a los tokens gastados para no pasar de 8000 por minuto
            time.sleep(response.usage.total_tokens / 8000 * 60)
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            print(f"Error en el intento {intento + 1}: {e}")
            time.sleep(60)
    return []


# Función que junta las reseñas de un lote con etiquetas para separarlas
def juntar_resenas(lote):
    return "\n".join(f'<resena id="{fila.Id}">\n{fila.Contenido}\n</resena>' for fila in lote.itertuples())


# Función que procesa un dataframe por lotes y devuelve una lista con un diccionario por reseña
def procesar(df, modelo, sys_prompt, tam_lote, max_tokens):
    resultados = []
    for i in range(0, len(df), tam_lote):
        lote = df.iloc[i:i + tam_lote]
        print(f"Lote {i // tam_lote + 1} de {-(-len(df) // tam_lote)}")
        respuesta = preguntar_groq(modelo, sys_prompt, juntar_resenas(lote), max_tokens)
        # A veces el modelo devuelve la lista sin la clave "resultados" o con elementos mal formados
        if isinstance(respuesta, dict):
            respuesta = respuesta.get("resultados", [])
        for item in respuesta:
            if isinstance(item, dict) and item.get("Id") in list(lote["Id"]):
                resultados.append(item)
    return resultados

### Prompt de filtrado

El prompt le dice al modelo que es un clasificador y explica cuándo una reseña es relevante (habla de aspectos concretos del juego) y cuándo no (spam, texto repetido, solo insultos o chistes). Las reseñas van separadas con etiquetas y el modelo contesta con un json con el Id, si es relevante y un motivo de una frase. Casi todas las reseñas están en inglés y sin puntuación, y el prompt avisa de que eso no es motivo para descartarlas.

In [8]:
sys_prompt_filtro = """Eres un clasificador de reseñas de videojuegos. Decides si cada reseña es RELEVANTE para extraer insights.

Una reseña es RELEVANTE (true) si menciona aspectos concretos del juego: jugabilidad, historia, gráficos, rendimiento, bugs, dificultad, precio, duración, contenido, comparaciones con otros juegos, etc., y permite identificar opiniones positivas o negativas útiles.

Una reseña NO es relevante (false) si es: spam, texto repetido o sin sentido, texto sin contenido semántico (solo insultos, solo emojis o palabras sueltas), o un meme, chiste o copypasta sin opinión real sobre el juego.

Notas:
- Las reseñas pueden estar en inglés o español, sin puntuación ni mayúsculas. Eso NO es motivo para descartarlas.
- Juzga solo el contenido, no te dejes influir por la longitud.

Responde SOLO con JSON válido, con este formato exacto (un elemento por reseña, mismo Id):
{"resultados": [{"Id": <int>, "relevante": true|false, "motivo": "<una frase en español>"}]}"""

In [9]:
df_top_100["Id"] = df_top_100.index

inicio = time.time()
resultados_filtro = procesar(df_top_100, modelo_filtro, sys_prompt_filtro, batch_size, 800)

# Si el modelo se ha dejado alguna reseña, se repite esas de una en una
faltan = df_top_100[~df_top_100["Id"].isin([r["Id"] for r in resultados_filtro])]
if len(faltan) > 0:
    resultados_filtro += procesar(faltan, modelo_filtro, sys_prompt_filtro, 1, 800)
print(f"Tiempo total: {(time.time() - inicio) / 60:.1f} minutos")

Lote 1 de 50


Error en el intento 1: Error code: 403 - {'error': {'message': 'Access denied. Please check your network settings.'}}


Error en el intento 2: Error code: 403 - {'error': {'message': 'Access denied. Please check your network settings.'}}


Error en el intento 3: Error code: 403 - {'error': {'message': 'Access denied. Please check your network settings.'}}


Lote 2 de 50


Error en el intento 1: Error code: 403 - {'error': {'message': 'Access denied. Please check your network settings.'}}


... (102 outputs left out)

Lote 22 de 24


Lote 23 de 24


Lote 24 de 24


Tiempo total: 58.5 minutos


In [10]:
df_filtro = pd.DataFrame(resultados_filtro)
df_filtrado = df_top_100.merge(df_filtro, on="Id")
df_relevantes = df_filtrado[df_filtrado["relevante"]].reset_index(drop=True)

print(f"Reseñas evaluadas: {len(df_filtrado)}, relevantes: {len(df_relevantes)}, descartadas: {len(df_filtrado) - len(df_relevantes)}")
df_filtrado[~df_filtrado["relevante"]][["Id", "Valoración", "longitud_caracteres", "motivo"]]

Reseñas evaluadas: 100, relevantes: 99, descartadas: 1


,Id,Valoración,longitud_caracteres,motivo
0,0,Recomendado,7972,"texto sin contenido semántico, solo repetición..."


In [11]:
df_relevantes.drop(columns=["relevante"]).to_csv("resultado_paso2_relevantes.csv", index=False)

## Paso 3: extracción de información estructurada con un segundo LLM

Con las reseñas relevantes, gpt-oss-120b devuelve un json por reseña con el esquema del enunciado y dos campos más del csv de ejemplo (ProblemasTecnicos y RelacionCalidadPrecio). El prompt pide la salida en español aunque la reseña esté en inglés, limita los valores de sentimiento, dificultad y calidad-precio, y le dice que ponga "No Mencionado" si algo no aparece para que no se invente nada.

El campo Recomendado no se lo pido al modelo, lo copio de la columna Valoración. Así no influye en el sentimiento y luego podemos comparar los dos.

In [12]:
sys_prompt_extraccion = """Eres un analista de reseñas de videojuegos. Para cada reseña extraes información estructurada.

REGLAS:
- Escribe SIEMPRE en español, aunque la reseña esté en inglés (traduce los aspectos).
- Usa solo información presente en la reseña, no inventes. Si un dato no aparece, usa "No Mencionado" (o lista vacía).
- Los aspectos son etiquetas breves de 2-5 palabras (por ejemplo "historia envolvente", "bugs frecuentes"), sin repetidos.
- Las reseñas pueden estar sin puntuación ni mayúsculas.

CAMPOS:
- SentimientoGeneral: "Positivo" | "Negativo" | "Neutral"
- AspectosPositivos: lista de aspectos positivos mencionados
- AspectosNegativos: lista de aspectos negativos mencionados
- Dificultad: "Demasiado Fácil" | "Fácil" | "Equilibrado" | "Difícil" | "Demasiado Difícil" | "No Mencionado"
- ProblemasTecnicos: lista de problemas técnicos (bugs, rendimiento, crashes, optimización), [] si no hay
- RelacionCalidadPrecio: "Buena" | "Regular" | "Mala" | "No Mencionado"

Responde SOLO con JSON válido, un elemento por reseña con su mismo Id:
{"resultados": [{"Id": <int>, "SentimientoGeneral": "...", "AspectosPositivos": [...], "AspectosNegativos": [...], "Dificultad": "...", "ProblemasTecnicos": [...], "RelacionCalidadPrecio": "..."}]}"""

In [13]:
inicio = time.time()
resultados_ext = procesar(df_relevantes, modelo_extraccion, sys_prompt_extraccion, batch_size, 2500)

# Si el modelo se ha dejado alguna reseña, se repite esas de una en una
faltan = df_relevantes[~df_relevantes["Id"].isin([r["Id"] for r in resultados_ext])]
if len(faltan) > 0:
    resultados_ext += procesar(faltan, modelo_extraccion, sys_prompt_extraccion, 1, 2500)
print(f"Tiempo total: {(time.time() - inicio) / 60:.1f} minutos")

Lote 1 de 50


Lote 2 de 50


Lote 3 de 50


Lote 4 de 50


Lote 5 de 50


Lote 6 de 50


... (47 outputs left out)

Lote 4 de 6


Lote 5 de 6


Lote 6 de 6


Tiempo total: 25.7 minutos


In [14]:
df_ext = pd.DataFrame(resultados_ext)
df_final = df_ext.merge(df_relevantes[["Id", "Valoración"]], on="Id").rename(columns={"Valoración": "Recomendado"})
df_final = df_final[["Id", "SentimientoGeneral", "AspectosPositivos", "AspectosNegativos", "Dificultad",
                     "ProblemasTecnicos", "RelacionCalidadPrecio", "Recomendado"]].sort_values("Id").reset_index(drop=True)
df_final.to_csv("resultado_final.csv", index=False)
df_final

,Id,SentimientoGeneral,AspectosPositivos,AspectosNegativos,Dificultad,ProblemasTecnicos,RelacionCalidadPrecio,Recomendado
0,1,Positivo,"[espacios urbanos detallados, verticalidad úni...","[dificultad demasiado fácil, falta de cierre e...",Fácil,[],Buena,Recomendado
1,2,Negativo,"[gráficos sólidos, ambiente cyberpunk, banda s...","[bugs y glitches frecuentes, crash ocasional, ...",No Mencionado,"[bugs gráficos, crash del juego, texturas falt...",Mala,No recomendado
2,3,Negativo,"[gráficos sólidos, ambiente cyberpunk atractiv...","[bugs y glitches gráficos, crash ocasional, mi...",Equilibrado,"[glitches gráficos, texturas faltantes, objeto...",Mala,No recomendado
3,4,Negativo,"[mundo hermoso, historia fascinante, personaje...","[dificultad excesivamente punitiva, sistema de...",Difícil,"[problemas de rendimiento, lag en combates, er...",Mala,No recomendado
4,5,Negativo,"[combate fluido, controles responsivos, animac...","[decisiones de diseño pobres, historia conserv...",Difícil,"[stutter de shaders, bajas tasas de frames, cr...",No Mencionado,No recomendado
...,...,...,...,...,...,...,...,...
94,95,Positivo,"[worldbuilding impresionante, tema futurista, ...","[actividades laterales repetitivas, bugs persi...",No Mencionado,[bugs que obligan a guardados rápidos],Regular,Recomendado
95,96,Negativo,"[línea argumental principal atrapante, algunas...","[glitches visuales y bugs, final apresurado y ...",No Mencionado,"[glitches visuales, bugs de rendimiento, probl...",Mala,Recomendado
96,97,Positivo,"[historia larga, gráficos fabulosos, música am...","[pocos bugs, problemas guardado ocasional, int...",Difícil,"[bugs menores, errores guardado, interfaz desa...",Buena,Recomendado
97,98,Negativo,"[mundo enorme, variedad de calabozos, algunas ...","[misiones repetitivas, historia débil, actores...",No Mencionado,[],Mala,No recomendado


## Análisis de resultados

In [15]:
# Sentimiento extraído por el LLM frente a la valoración del usuario
pd.crosstab(df_final["SentimientoGeneral"], df_final["Recomendado"])

Recomendado,No recomendado,Recomendado
SentimientoGeneral,,
Negativo,62,4
Neutral,2,3
Positivo,1,27


In [16]:
print(df_final["Dificultad"].value_counts())
print()
print(df_final["RelacionCalidadPrecio"].value_counts())

Dificultad
No Mencionado      67
Difícil            21
Equilibrado         7
Fácil               2
Demasiado Fácil     2
Name: count, dtype: int64

RelacionCalidadPrecio
Mala             46
No Mencionado    31
Buena            14
Regular           8
Name: count, dtype: int64


In [17]:
from collections import Counter

for columna in ["AspectosPositivos", "AspectosNegativos", "ProblemasTecnicos"]:
    print(f"Top 10 de {columna}")
    print(Counter(a.lower() for lista in df_final[columna] for a in lista).most_common(10))
    print()

Top 10 de AspectosPositivos
[('banda sonora excelente', 13), ('gráficos impresionantes', 13), ('historia envolvente', 9), ('personajes interesantes', 7), ('personajes bien desarrollados', 6), ('misiones secundarias interesantes', 5), ('variedad de armas', 4), ('personajes memorables', 4), ('mundo hermoso', 4), ('atmósfera inmersiva', 4)]

Top 10 de AspectosNegativos
[('bugs frecuentes', 14), ('personalización limitada', 13), ('optimización deficiente', 10), ('misiones repetitivas', 8), ('optimización pobre', 7), ('ia deficiente', 7), ('problemas de rendimiento', 5), ('ia pobre', 4), ('muchos bugs', 4), ('conducción deficiente', 4)]

Top 10 de ProblemasTecnicos
[('bugs', 25), ('optimización deficiente', 15), ('glitches', 10), ('problemas de rendimiento', 9), ('crashes', 7), ('bugs visuales', 6), ('bugs frecuentes', 5), ('caídas de fps', 5), ('bugs varios', 5), ('bajo rendimiento', 5)]



## Conclusiones

El filtro del paso 2 casi no descarta reseñas: solo una de las 100 (por ser texto repetido). Tiene sentido, porque al quedarnos con las 100 más largas casi todas tienen contenido real, y es una limitación de partir de ese top 100. En la ejecución, la API devolvió errores 403 en los primeros lotes del filtro. El código los reintentó y después repitió de una en una las reseñas que faltaban, así que al final se evaluaron las 100.

Con las 99 reseñas extraídas, el sentimiento que da el modelo coincide con la valoración del usuario en 89 casos, 5 salen neutrales y hay 5 contradicciones. Lo que más se valora es la banda sonora, los gráficos y la historia, y lo que más se critica son los bugs, la personalización limitada y la optimización. La dificultad casi no se menciona (67 de 99) y la relación calidad-precio se valora como mala en 46 reseñas.